In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Install required libraries (run once per Kaggle session)
!pip install vaderSentiment nrclex --quiet

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import re
import warnings
warnings.filterwarnings('ignore')

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from nrclex import NRCLex

# Confirm your files are there
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Load business and review JSON files line by line (they're large)
def load_json_lines(filepath, max_rows=None):
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows:
                break
            records.append(json.loads(line))
    return pd.DataFrame(records)

BUSINESS_PATH = '/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_business.json'
REVIEW_PATH   = '/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_review.json'

print("Loading business data...")
df_business = load_json_lines(BUSINESS_PATH)
print(f"Businesses loaded: {len(df_business):,}")

print("Loading review data (this may take a minute)...")
df_review = load_json_lines(REVIEW_PATH, max_rows=500_000)  # cap for memory
print(f"Reviews loaded:    {len(df_review):,}")

In [ ]:
print("=== BUSINESS COLUMNS ===")
print(df_business.columns.tolist())
print(df_business[['name','city','state','stars','categories']].head(3))

print("\n=== REVIEW COLUMNS ===")
print(df_review.columns.tolist())
print(df_review[['stars','text','business_id']].head(3))

print("\n=== CITIES IN DATASET ===")
print(df_business['city'].value_counts().head(20))

In [ ]:
# Approximate cost-of-living score per city (1=low, 10=high)
# Based on Census median household income tiers as described in your proposal
COL_SCORES = {
    # High cost (HCOL) — score 7-10
    'Santa Barbara':  9.0, 'Philadelphia': 7.5, 'Scottsdale': 7.0,
    'Las Vegas':      6.5, 'Denver': 6.8, 'Portland': 7.2,
    # Low cost (LCOL) — score 1-5
    'Nashville':  5.0, 'Tampa': 4.5, 'Charlotte': 4.8,
    'Cleveland':  3.5, 'Pittsburgh': 3.8, 'Indianapolis': 3.5,
    'Boise':      4.0, 'Tucson': 3.2,
}

HCOL_CITIES = [c for c, s in COL_SCORES.items() if s >= 6.5]
LCOL_CITIES = [c for c, s in COL_SCORES.items() if s < 6.5]

print("HCOL cities:", HCOL_CITIES)
print("LCOL cities:", LCOL_CITIES)

# Add cost tier labels
df_business['col_score'] = df_business['city'].map(COL_SCORES)
df_business['col_tier']  = df_business['col_score'].apply(
    lambda s: 'HCOL' if pd.notna(s) and s >= 6.5 else ('LCOL' if pd.notna(s) else None)
)
print(df_business['col_tier'].value_counts())

In [ ]:
# Define category keywords
FAST_FOOD_KEYWORDS  = ['Fast Food', 'Burgers', 'Pizza', 'Sandwiches', 'Chicken Wings']
FINE_DINING_KEYWORDS = ['Fine Dining', 'Steakhouses', 'French', 'Seafood', 'Japanese']

def classify_category(cats):
    if pd.isna(cats):
        return None
    if any(k in cats for k in FAST_FOOD_KEYWORDS):
        return 'Fast Food'
    if any(k in cats for k in FINE_DINING_KEYWORDS):
        return 'Fine Dining'
    return None

df_business['dining_category'] = df_business['categories'].apply(classify_category)

# Also flag franchise vs local (basic heuristic — expand list as needed)
FRANCHISE_NAMES = ['McDonald', 'Subway', 'Taco Bell', 'Burger King', 'Wendy', 
                   'KFC', 'Chick-fil-A', 'Domino', 'Pizza Hut', "Denny's", 'IHOP']

df_business['ownership_type'] = df_business['name'].apply(
    lambda n: 'Franchise' if any(f in str(n) for f in FRANCHISE_NAMES) else 'Local'
)

# Keep only businesses we care about in our study cities
df_biz_filtered = df_business[
    df_business['col_tier'].notna() &
    df_business['dining_category'].notna()
].copy()

print(f"Filtered businesses: {len(df_biz_filtered):,}")
print(df_biz_filtered.groupby(['col_tier','dining_category','ownership_type']).size())

In [ ]:
df_merged = df_review.merge(
    df_biz_filtered[['business_id','city','state','col_score','col_tier',
                     'dining_category','ownership_type','attributes']],
    on='business_id',
    how='inner'
)

print(f"Merged review rows: {len(df_merged):,}")
print(df_merged[['stars','text','city','col_tier','dining_category']].head(3))

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+', '', text)           # remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)         # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()       # collapse whitespace
    return text

df_merged['clean_text'] = df_merged['text'].apply(clean_text)
print("Sample cleaned text:")
print(df_merged['clean_text'].iloc[0][:200])

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_vader_compound(text):
    return analyzer.polarity_scores(text)['compound']  # range: -1 to +1

print("Running VADER sentiment scoring...")
df_merged['vader_compound'] = df_merged['clean_text'].apply(get_vader_compound)

# Normalize star rating to 0-1 scale (stars go 1-5)
df_merged['stars_norm']  = (df_merged['stars'] - 1) / 4
# Normalize VADER compound to 0-1 scale (compound goes -1 to +1)
df_merged['vader_norm']  = (df_merged['vader_compound'] + 1) / 2

# Core engineered feature: Sentiment-Rating Gap
df_merged['sentiment_rating_gap'] = df_merged['stars_norm'] - df_merged['vader_norm']

print(df_merged[['stars','vader_compound','stars_norm','vader_norm','sentiment_rating_gap']].describe())

In [ ]:
# Download NRC lexicon directly - more reliable than NRCLex package
!wget -q https://raw.githubusercontent.com/dinbav/LeXmo/master/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt -O /kaggle/working/nrc_lexicon.txt

# Build lexicon dictionary manually
nrc_lexicon = {}
with open('/kaggle/working/nrc_lexicon.txt', 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 3:
            word, emotion, flag = parts
            if flag == '1':
                if word not in nrc_lexicon:
                    nrc_lexicon[word] = []
                nrc_lexicon[word].append(emotion)

print(f"Lexicon loaded: {len(nrc_lexicon)} words")
print("Sample:", list(nrc_lexicon.items())[:5])

# Test it
test_text = "I am so happy and excited about this wonderful food"
test_words = test_text.lower().split()
test_emotions = {}
for word in test_words:
    for emotion in nrc_lexicon.get(word, []):
        test_emotions[emotion] = test_emotions.get(emotion, 0) + 1
print("Test emotions:", test_emotions)

In [ ]:
EMOTIONS = ['anger','fear','anticipation','trust','surprise','sadness','joy','disgust']

def get_emotions(text):
    if not text or not isinstance(text, str) or not text.strip():
        return {e: 0.0 for e in EMOTIONS}
    try:
        words = text.lower()[:1000].split()
        counts = {e: 0 for e in EMOTIONS}
        for word in words:
            for emotion in nrc_lexicon.get(word, []):
                if emotion in counts:
                    counts[emotion] += 1
        total = sum(counts.values()) or 1
        return {e: counts[e] / total for e in EMOTIONS}
    except Exception:
        return {e: 0.0 for e in EMOTIONS}

print("Running emotion analysis...")
emotion_scores = df_merged['text'].apply(get_emotions)
emotion_df = pd.DataFrame(emotion_scores.tolist(), index=df_merged.index)

df_merged = pd.concat([df_merged.drop(columns=EMOTIONS, errors='ignore'), emotion_df], axis=1)
df_merged['dominant_emotion'] = emotion_df[EMOTIONS].idxmax(axis=1)

print("Done! Sample:")
print(df_merged[['stars', 'vader_compound', 'dominant_emotion'] + EMOTIONS].head(5))

In [ ]:
# Yelp price tier from business attributes ($ to $$$$)
def get_price_tier(attrs):
    try:
        return attrs.get('RestaurantsPriceRange2', None) if isinstance(attrs, dict) else None
    except:
        return None

df_merged['price_tier'] = df_business.set_index('business_id').loc[
    df_merged['business_id'], 'attributes'
].values
df_merged['price_tier'] = df_merged['price_tier'].apply(get_price_tier)

# Select final columns your teammates need for Phase 3
FINAL_COLS = [
    'business_id','city','state',
    'col_score','col_tier',
    'dining_category','ownership_type','price_tier',
    'stars','stars_norm',
    'vader_compound','vader_norm',
    'sentiment_rating_gap',
    'dominant_emotion',
    'clean_text',
] + EMOTIONS

df_final = df_merged[FINAL_COLS].copy()

# Save for Phase 3 team
df_final.to_csv('/kaggle/working/yelp_processed.csv', index=False)
print(f"Saved: /kaggle/working/yelp_processed.csv  ({len(df_final):,} rows)")

print("\n=== QUICK SUMMARY ===")
print(df_final.groupby(['col_tier','dining_category'])['sentiment_rating_gap'].agg(['mean','std','count']))
print("\nDominant emotion distribution:")
print(df_final['dominant_emotion'].value_counts())

In [ ]:
!pip install -q -U transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL = "Qwen/Qwen3.5-2B"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("Qwen3 1.7B ready in float16.")

In [ ]:
print("done")

In [ ]:
import pandas as pd
import json, re, torch, time
from tqdm.auto import tqdm

print("Loading dataset...")
df = pd.read_csv('/kaggle/working/yelp_processed.csv')
print(f"Total rows loaded: {len(df):,}")
print("\nFirst 3 rows of text to verify:")
for i, text in enumerate(df['clean_text'].head(3)):
    print(f"{i+1}. {text[:100]}...")

start_time = time.time()
BATCH_SIZE = 32

# LEFT padding — critical for batched generation
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompt(text):
    messages = [
        {"role": "system", "content": "You are a JSON API. Output only valid JSON. No words. No explanation. No markdown."},
        {"role": "user", "content": f"""Sentiment of this restaurant review for food, service, price.
Review: {str(text)[:300]}
Output ONLY this JSON:
{{"f": X, "s": X, "p": X}}
Where X is:
-1.0 = clearly negative
0.0  = mixed or neutral  
1.0  = clearly positive
null = not mentioned (DEFAULT IF YOU ARE UNSURE)
Examples:
{{"f": 1.0, "s": -1.0, "p": null}}
{{"f": 0.0, "s": null,  "p": -1.0}}
{{"f": 1.0, "s": 1.0,  "p": 1.0}}"""}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

def parse(raw):
    raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    match = re.search(r'\{[^}]+\}', raw)
    if not match:
        print(f"  [raw output]: {raw[:100]}")
        return {"f": "N/A", "s": "N/A", "p": "N/A"}
    try:
        parsed = json.loads(match.group())
        return {
            "f": parsed.get("f", "N/A"),
            "s": parsed.get("s", "N/A"),
            "p": parsed.get("p", "N/A")
        }
    except:
        return {"f": "N/A", "s": "N/A", "p": "N/A"}

def grade_batch(texts):
    prompts = [build_prompt(t) for t in texts]
    inputs  = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    results = []
    for i, seq in enumerate(out):
        # slice off input tokens correctly per sequence
        input_len = inputs['input_ids'].shape[1]
        raw = tokenizer.decode(
            seq[input_len:],
            skip_special_tokens=True
        )
        results.append(parse(raw))
    return results

results = []
texts   = df['clean_text'].tolist()
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Grading"):
    batch = texts[i : i + BATCH_SIZE]
    results.extend(grade_batch(batch))

scores_df = pd.DataFrame(results).rename(columns={
    "f": "qwen_food",
    "s": "qwen_service",
    "p": "qwen_price"
})

df_final = pd.concat([df.reset_index(drop=True), scores_df], axis=1)
df_final.to_csv('/kaggle/working/yelp_qwen_scored.csv', index=False)

elapsed = time.time() - start_time
rows_per_sec = len(df_final) / elapsed
print(f"\nDone in {elapsed:.1f}s ({rows_per_sec:.1f} rows/sec) — saved {len(df_final):,} rows.")
print(df_final[['clean_text','qwen_food','qwen_service','qwen_price']])

In [ ]:
# Statistical Analysis

In [ ]:
!pip install pingouin
import pandas as pd
import numpy as np
import pingouin as pg
import matplotlib.pyplot as plt
import seaborn as sns

# Load the scored dataset containing the sentiment-rating gaps and LLM scores
df = pd.read_csv('/kaggle/working/yelp_qwen_scored.csv') # Takes about 10 hours


In [ ]:
# Two-Way ANOVA Test - this will check if cost-of-living and dining category interact to affect the sentiment-rating gap
# This is to prove that the gap isn't random and depends on the specific interaction between geographic location and dining category

# Drop any rows with missing values in the target cols
df_anova = df.dropna(subset=['sentiment_rating_gap', 'col_tier', 'dining_category'])

print("Two-Way ANOVA Results:")
anova_results = pg.anova(dv='sentiment_rating_gap', between=['col_tier', 'dining_category'], data=df_anova)
print(anova_results)

In [ ]:
# Boxplot Visualization for Two-Way ANOVA - compares fine dining and fast food in HCOL vs. LCOL cities to see where the sentiment-rating gap is the widest

plt.figure(figsize=(10, 6))
sns.boxplot(x='col_tier', y='sentiment_rating_gap', hue='dining_category', data=df_anova, palette='Set2')

plt.title('Sentiment-Rating Gap by Cost of Living and Dining Category')
plt.xlabel('Cost of Living Tier')
plt.ylabel('Sentiment-Rating Gap')
plt.show()

In [ ]:
# Multiple Linear Regression Test - tests how much food, service, and price each contribute to the sentiment-rating gap
# This is to see which of these three factors is the strongest driver of customer ratings

# Drop any rows with missing values in the target cols
df_linreg = df.dropna(subset=['sentiment_rating_gap', 'qwen_food', 'qwen_service', 'qwen_price'])

print("Multiple Linear Regression Results:")
X = df_linreg[['qwen_food', 'qwen_service', 'qwen_price']] # Independent variables
y = df_linreg['sentiment_rating_gap'] # Dependent variable

linreg_results = pg.linear_regression(X, y)
print(linreg_results)

In [ ]:
# Barplot Linear Regression Visualization - this visualizes the impact of food, service, and price on ratings
# The factor with the largest bar has the highest influence on ratings

plt.figure(figsize=(10,6))
coef_df = linreg_results[linreg_results['names'] != 'Intercept'] # Removes the intercept row to focus on the three main features (food, service, price)

sns.barplot(x='coef', y='names', data=coef_df)

plt.errorbar(x=coef_df['coef'], y=coef_df['names'], xerr=coef_df['se'], fmt='none', c='black', capsize=5) # Shows CIs
plt.title('Impact of Review Factors on Sentiment-Rating Gap')
plt.xlabel('Regression Coeffecient (Effect Size)')
plt.ylabel('Review Factor')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import shap
import warnings
warnings.filterwarnings('ignore')

# 1. Load data 
path = "/kaggle/working/yelp_qwen_scored.csv" # Takes about 10 hours
df = pd.read_csv(path)

# 2. Preprocessing 
features = ['qwen_food', 'qwen_service', 'qwen_price', 'vader_compound']
df[features] = df[features].fillna(0)

hcol_df = df[df['col_tier'] == 'HCOL'].copy()
lcol_df = df[df['col_tier'] == 'LCOL'].copy()
scenarios = [(df, "OVERALL"), (hcol_df, "HCOL"), (lcol_df, "LCOL")]

MIN_SAMPLE = 30  # guard for tiny segments


# 3. Helper for Partial R² (dominance analysis approximation) 
def partial_r2(model_full, X_const, y, feature_names):
    """Drop one feature at a time; measure R² drop = partial R² for that feature."""
    r2_full = model_full.rsquared
    partials = {}
    for feat in feature_names:
        cols_reduced = [c for c in X_const.columns if c != feat]
        reduced_model = sm.OLS(y, X_const[cols_reduced]).fit()
        partials[feat] = max(0.0, r2_full - reduced_model.rsquared)
    return partials


# 4. Main analysis function 
def run_deep_analysis(data, title):
    if len(data) < MIN_SAMPLE:
        print(f"\n[SKIP] {title}: only {len(data)} rows — below minimum ({MIN_SAMPLE}).")
        return

    print(f"\n{'='*80}")
    print(f"  STATISTICAL REPORT: {title}  (N={len(data):,})")
    print(f"{'='*80}")

    X = data[features].reset_index(drop=True)
    y = data['stars'].reset_index(drop=True)

    # Descriptive statistics 
    print("\n[1] DESCRIPTIVE STATISTICS")
    print(pd.concat([X, y], axis=1).describe().round(3).to_string())

    # Regression (unstandardised) 
    X_const = sm.add_constant(X, has_constant='add')
    model = sm.OLS(y, X_const).fit()

    # Standardised beta weights 
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X), columns=features
    )
    X_scaled_const = sm.add_constant(X_scaled, has_constant='add')
    beta_model = sm.OLS(y, X_scaled_const).fit()

    # VIF (skip intercept row) 
    vif_values = {
        col: variance_inflation_factor(X_const.values, i)
        for i, col in enumerate(X_const.columns)
        if col != 'const'
    }

    # Partial R² 
    part_r2 = partial_r2(model, X_const, y, features)

    # Combined regression table 
    conf = model.conf_int()
    conf.columns = ['CI_2.5%', 'CI_97.5%']

    results = pd.DataFrame({
        'Unstd. Coef':     model.params,
        'Std. Beta':       [np.nan] + list(beta_model.params[1:]),   
        'P-Value':         model.pvalues,
        'CI 2.5%':         conf['CI_2.5%'],
        'CI 97.5%':        conf['CI_97.5%'],
        'VIF':             [np.nan] + [vif_values[f] for f in features],
        'Partial R²':      [np.nan] + [part_r2[f] for f in features],
    })

    print("\n[2] REGRESSION RESULTS")
    print(results.round(4).to_string())
    print(f"\n  Adjusted R²  : {model.rsquared_adj:.4f}")
    print(f"  F-stat p-val : {model.f_pvalue:.4e}")

    # Diagnostic Tests 
    print("\n[3] DIAGNOSTIC TESTS")

    # Breusch-Pagan (heteroscedasticity)
    _, bp_p, _, _ = het_breuschpagan(model.resid, X_const)
    bp_flag = "⚠ DETECTED — errors are non-uniform across rating bands." if bp_p < 0.05 else "✓ Not detected."
    print(f"  Breusch-Pagan (heteroscedasticity)  p={bp_p:.4e}  →  {bp_flag}")

    # Jarque-Bera (residual normality)
    jb_stat, jb_p = stats.jarque_bera(model.resid)
    jb_flag = "⚠ Residuals are non-normal — interpret p-values cautiously." if jb_p < 0.05 else "✓ Residuals approximately normal."
    print(f"  Jarque-Bera  (normality)            p={jb_p:.4e}  →  {jb_flag}")

    # Durbin-Watson (autocorrelation)
    from statsmodels.stats.stattools import durbin_watson
    dw = durbin_watson(model.resid)
    dw_flag = "✓ No strong autocorrelation." if 1.5 < dw < 2.5 else "⚠ Possible autocorrelation."
    print(f"  Durbin-Watson (autocorrelation)     DW={dw:.3f}            →  {dw_flag}")

    # SHAP & Random Forest importance 
    print("\n[4] FEATURE IMPORTANCE (SHAP + Permutation)")

    n_sample = min(2000, len(X))
    sample_idx = X.sample(n=n_sample, random_state=42).index
    X_sample = X.loc[sample_idx]
    y_sample = y.loc[sample_idx]

    rf = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)
    rf.fit(X_sample, y_sample)

    # SHAP values
    explainer = shap.TreeExplainer(rf)
    shap_vals = explainer.shap_values(X_sample)
    if isinstance(shap_vals, list):          
        shap_vals = shap_vals[0]

    mean_abs_shap = np.abs(shap_vals).mean(axis=0)

    # Permutation importance (model-agnostic cross-check)
    perm = permutation_importance(rf, X_sample, y_sample, n_repeats=10, random_state=42)

    importance_df = pd.DataFrame({
        'Feature':        features,
        'Mean |SHAP|':    mean_abs_shap,
        'Perm Importance': perm.importances_mean,
    }).sort_values('Mean |SHAP|', ascending=False)
    print(importance_df.round(4).to_string(index=False))

    # Plots 
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    fig.suptitle(f"Diagnostic Plots - {title}", fontsize=14, fontweight='bold')

    # Correlation matrix
    sns.heatmap(
        pd.concat([X, y], axis=1).corr(),
        annot=True, fmt='.2f', cmap='coolwarm',
        ax=axes[0], linewidths=0.5
    )
    axes[0].set_title("Correlation Matrix")

    # Residuals vs. fitted
    fitted = model.fittedvalues
    axes[1].scatter(fitted, model.resid, alpha=0.3, s=10, color='steelblue')
    axes[1].axhline(0, color='red', linewidth=1.2, linestyle='--')
    axes[1].set_xlabel("Fitted Values")
    axes[1].set_ylabel("Residuals")
    axes[1].set_title("Residuals vs Fitted")
    # LOWESS trend
    lowess = sm.nonparametric.lowess(model.resid, fitted, frac=0.3)
    axes[1].plot(lowess[:, 0], lowess[:, 1], color='orange', linewidth=2, label='LOWESS')
    axes[1].legend()

    # Q-Q plot
    sm.qqplot(model.resid, line='s', ax=axes[2], alpha=0.4)
    axes[2].set_title("Q-Q Plot (Residual Normality)")

    plt.tight_layout()
    plt.show()

    # SHAP summary (bee swarm)
    plt.figure(figsize=(10, 5))
    shap.summary_plot(shap_vals, X_sample, show=False)
    plt.title(f"SHAP Directional Impact - {title}", fontsize=13)
    plt.tight_layout()
    plt.show()

    # SHAP dependence plot for top feature
    top_feature = importance_df['Feature'].iloc[0]
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(top_feature, shap_vals, X_sample, show=False)
    plt.title(f"SHAP Dependence: {top_feature} — {title}", fontsize=13)
    plt.tight_layout()
    plt.show()

    print(f"\n{'─'*80}\n")


# 5. All-features stepwise R² analysis (Overall only) 
def run_feature_addition_analysis(data, title="OVERALL"):
    """
    Two complementary views of how each feature contributes to Adjusted R²:

    A) Forward stepwise - add features one at a time (best-first by Adj R²),
       showing the cumulative gain at each step.

    B) All-subsets single-feature -> full-model table - every feature alone,
       then all four together, so you can see isolated vs. joint explanatory power.
    """
    print(f"\n{'='*80}")
    print(f"  FEATURE CONTRIBUTION TO ADJ R²: {title}  (N={len(data):,})")
    print(f"{'='*80}")

    # Detect every numeric column except the target and known ID-like cols
    # 'stars' is y — explicitly excluded so it never appears as a predictor
    exclude = {'stars', 'col_tier'}
    all_numeric = [
        c for c in data.select_dtypes(include=[np.number]).columns
        if c not in exclude
    ]
    assert 'stars' not in all_numeric, "BUG: 'stars' leaked into predictor list!"
    # Fill NAs so regressions don't drop rows silently
    data_clean = data[all_numeric + ['stars']].fillna(0).reset_index(drop=True)
    y = data_clean['stars']

    # Single-feature baselines 
    print("\n[A] SINGLE-FEATURE REGRESSIONS  (each feature alone vs stars)")
    single_rows = []
    for feat in all_numeric:
        xc = sm.add_constant(data_clean[[feat]], has_constant='add')
        m  = sm.OLS(y, xc).fit()
        single_rows.append({
            'Feature':    feat,
            'Coef':       m.params[feat],
            'P-Value':    m.pvalues[feat],
            'R²':         m.rsquared,
            'Adj R²':     m.rsquared_adj,
        })
    single_df = (
        pd.DataFrame(single_rows)
          .sort_values('Adj R²', ascending=False)
          .reset_index(drop=True)
    )
    print(single_df.round(4).to_string(index=False))

    # Forward stepwise by Adj R² 
    print("\n[B] FORWARD STEPWISE - cumulative Adj R² as features are added")
    remaining   = list(all_numeric)
    chosen      = []
    step_rows   = []
    prev_adj_r2 = 0.0

    while remaining:
        best_feat    = None
        best_adj_r2  = -np.inf
        for feat in remaining:
            candidate = chosen + [feat]
            xc = sm.add_constant(data_clean[candidate], has_constant='add')
            m  = sm.OLS(y, xc).fit()
            if m.rsquared_adj > best_adj_r2:
                best_adj_r2 = m.rsquared_adj
                best_feat   = feat

        # Stop early if adding any feature hurts Adj R²
        if best_adj_r2 < prev_adj_r2:
            print(f"  -> Early stop: no remaining feature improves Adj R² "
                  f"(best candidate would drop it to {best_adj_r2:.4f})")
            break

        chosen.append(best_feat)
        remaining.remove(best_feat)
        gain = best_adj_r2 - prev_adj_r2
        step_rows.append({
            'Step':          len(chosen),
            'Feature Added': best_feat,
            'Features So Far': ', '.join(chosen),
            'Adj R²':        best_adj_r2,
            'Δ Adj R²':      gain,
        })
        prev_adj_r2 = best_adj_r2

    step_df = pd.DataFrame(step_rows)
    print(step_df.round(4).to_string(index=False))

    # Full model with all features 
    xc_full = sm.add_constant(data_clean[all_numeric], has_constant='add')
    m_full  = sm.OLS(y, xc_full).fit()
    print(f"\n[C] FULL MODEL ({len(all_numeric)} features)")
    print(f"  Adj R²   : {m_full.rsquared_adj:.4f}")
    print(f"  R²       : {m_full.rsquared:.4f}")
    print(f"  F p-val  : {m_full.f_pvalue:.4e}")

    full_coef = pd.DataFrame({
        'Coef':    m_full.params,
        'P-Value': m_full.pvalues,
    }).drop('const').sort_values('P-Value')
    print(full_coef.round(4).to_string())

    # Plots 
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(f"Feature Contribution Analysis — {title}", fontsize=14, fontweight='bold')

    # Bar chart: single-feature Adj R²
    colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in single_df['Adj R²']]
    axes[0].barh(single_df['Feature'], single_df['Adj R²'], color=colors)
    axes[0].axvline(0, color='black', linewidth=0.8)
    axes[0].set_xlabel("Adj R² (alone)")
    axes[0].set_title("Single-Feature Explanatory Power")
    axes[0].invert_yaxis()
    for i, (val, feat) in enumerate(zip(single_df['Adj R²'], single_df['Feature'])):
        axes[0].text(max(val, 0) + 0.001, i, f"{val:.4f}", va='center', fontsize=8)

    # Step chart: cumulative Adj R² from forward stepwise
    if not step_df.empty:
        axes[1].plot(step_df['Step'], step_df['Adj R²'],
                     marker='o', linewidth=2.5, color='steelblue', markersize=8)
        for _, row in step_df.iterrows():
            axes[1].annotate(
                f"+{row['Δ Adj R²']:.4f}\n({row['Feature Added']})",
                xy=(row['Step'], row['Adj R²']),
                xytext=(5, 5), textcoords='offset points', fontsize=8
            )
        axes[1].set_xticks(step_df['Step'])
        axes[1].set_xlabel("Step (feature added)")
        axes[1].set_ylabel("Cumulative Adj R²")
        axes[1].set_title("Forward Stepwise: Cumulative Adj R²")
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(f"\n{'-'*80}\n")


# 6. Run
# Run full deep analysis for all segments
for target_df, target_title in scenarios:
    run_deep_analysis(target_df, target_title)

# Run the all-features R² contribution analysis on overall only
run_feature_addition_analysis(df, title="OVERALL")